In [164]:
from numpy import select
import pandas as pd
import duckdb
pd.set_option('display.max_columns', None)

chemin_fichier = r"retailcorps.parquet"

df=duckdb.query(f"""
    select *
    From read_parquet('{chemin_fichier}')
""").df()

In [165]:
df.head(5)

,OrderLineID,Invoice,StockCode,Description,Quantity,InvoiceDate,CustomerID,Country,ProductCategoryCode,ProductCategory,ProductFamily,ProductNameClean,UnitPriceGBP,UnitCostGBP,CustomerSegment,CustomerType,LoyaltyProgram,SalesChannel,PaymentMethod,WarehouseID,PromotionCode,DiscountRate,OrderStatus,DeliveryTime,Continent,Region,Currency
0,1,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,13085.0,United Kingdom,SEASONAL,Seasonal,Occasions,15Cm Christmas Glass Ball 20 Lights,17.82,8.47,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
1,2,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,13085.0,United Kingdom,HOME,Home decoration,Interior,Pink Cherry Lights,21.29,11.43,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
2,3,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,13085.0,United Kingdom,HOME,Home decoration,Interior,White Cherry Lights,7.2,2.6,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
3,4,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,13085.0,United Kingdom,HOME,Home decoration,Interior,"Record Frame 7"" Single Size",21.15,11.45,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
4,5,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,13085.0,United Kingdom,OTHER,Other,Other,Strawberry Ceramic Trinket Box,12.74,5.85,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP


In [166]:
import duckdb

duckdb.register('retail_raw', df)

In [167]:
duckdb.sql("""
    SELECT COUNT(*) AS nb_lignes
    FROM retail_raw
""").df()

,nb_lignes
0,1072707


In [168]:
# Décrire les structures de la table: liste des colonnes et leur type actuel (VARCHAR pour toutes, car chargées en mode textepour l'audit)
# VARCHAR veut dire que la donnée est stockée comme du texte, même si elle ressemble à un nombre ou une date.
duckdb.sql("""
    DESCRIBE retail_raw
""").df()

,column_name,column_type,null,key,default,extra
0,OrderLineID,VARCHAR,YES,None,None,None
1,Invoice,VARCHAR,YES,None,None,None
2,StockCode,VARCHAR,YES,None,None,None
3,Description,VARCHAR,YES,None,None,None
4,Quantity,VARCHAR,YES,None,None,None
5,InvoiceDate,VARCHAR,YES,None,None,None
6,CustomerID,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,ProductCategoryCode,VARCHAR,YES,None,None,None
9,ProductCategory,VARCHAR,YES,None,None,None


Audit — Structure des colonnes (DESCRIBE)

Résultat : La table `retail_raw` contient 27 colonnes, toutes actuellement typées en `VARCHAR` (texte), car le chargement initial a volontairement conservé tout en texte brut afin de permettre un audit sans perte ni erreur de conversion.

Colonnes identifiées, par catégorie :
- Identifiants : `OrderLineID`, `Invoice`, `CustomerID`
- Produit : `StockCode`, `Description`, `ProductCategoryCode`, `ProductCategory`, `ProductFamily`, `ProductNameClean`
- Transaction : `Quantity`, `InvoiceDate`, `UnitPriceGBP`, `UnitCostGBP`
- Client : `Country`, `CustomerSegment`, `CustomerType`, `LoyaltyProgram`
- Vente : `SalesChannel`, `PaymentMethod`, `WarehouseID`, `PromotionCode`



In [169]:
# Afficher Invoice, InvoiceDate, SalesChannel et PaymentMethod pour explorer les données
duckdb.sql("""
    SELECT Invoice, InvoiceDate, SalesChannel, PaymentMethod
    FROM retail_raw
    LIMIT 20
""").df()

,Invoice,InvoiceDate,SalesChannel,PaymentMethod
0,489434,2009-12-01 07:45:00,Online,Card
1,489434,2009-12-01 07:45:00,Online,Card
2,489434,2009-12-01 07:45:00,Online,Card
3,489434,2009-12-01 07:45:00,Online,Card
4,489434,2009-12-01 07:45:00,Online,Card
5,489434,2009-12-01 07:45:00,Online,Card
6,489434,2009-12-01 07:45:00,Online,Card
7,489434,2009-12-01 07:45:00,Online,Card
8,489435,2009-12-01 07:46:00,Phone,Invoice
9,489435,2009-12-01 07:46:00,Phone,Invoice


In [170]:
# Voir les différentes valeurs de SalesChannel et leur fréquence
duckdb.sql("""
    SELECT SalesChannel, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY SalesChannel
    ORDER BY nb DESC
""").df()

,SalesChannel,nb
0,Marketplace,271975
1,Online,271479
2,Store,264177
3,Phone,260881
4,None,4195


In [171]:
# Compter combien de lignes ont SalesChannel = 'NONE'
duckdb.sql("""
    SELECT COUNT(*) AS nb_none
    FROM retail_raw
    WHERE SalesChannel = 'None'
""").df()

,nb_none
0,0


In [172]:
# Confirmer que ce sont bien des vraies cases vides (NULL) et pas le mot "None"
duckdb.sql("""
    SELECT COUNT(*) AS nb_vraiment_vide
    FROM retail_raw
    WHERE SalesChannel IS NULL
""").df()

,nb_vraiment_vide
0,4195


In [173]:
# Voir les différentes valeurs de PaymentMethod et leur fréquence
duckdb.sql("""
    SELECT PaymentMethod, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY PaymentMethod
    ORDER BY nb DESC
""").df()

,PaymentMethod,nb
0,PayPal,273029
1,Card,272611
2,Bank transfer,265183
3,Invoice,261884


Conclusion — Invoice

Le fichier contient 1 072 707 lignes, mais seulement 53 628 factures différentes.

Cela s'explique simplement : une personne peut acheter plusieurs produits en une seule commande. Par exemple, si un client achète 5 produits différents, cela crée 1 seule facture (Invoice), mais 5 lignes dans le fichier (1 ligne par produit).

Aucune ligne n'a de numéro de facture manquant (0 valeur manquante).

In [174]:
# Vérifier les codes promo : quelles valeurs existent et combien de fois chacune apparaît
duckdb.sql("""
    SELECT PromotionCode, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY PromotionCode
    ORDER BY nb DESC
""").df()

,PromotionCode,nb
0,SPRING,222852
1,NONE,214157
2,WELCOME10,213557
3,XMAS,212674
4,CLEARANCE,207690
5,None,1777


In [175]:
duckdb.sql("""
    SELECT 
        CASE WHEN PromotionCode = 'NONE' OR PromotionCode IS NULL THEN 'NONE' ELSE PromotionCode END AS PromotionCode_clean,
        COUNT(*) AS nb
    FROM retail_raw
    GROUP BY PromotionCode_clean
    ORDER BY nb DESC
""").df()

,PromotionCode_clean,nb
0,SPRING,222852
1,NONE,215934
2,WELCOME10,213557
3,XMAS,212674
4,CLEARANCE,207690


Regrouper les deux colonnes none dans une seule afin que ça soit plus clair 

In [176]:
# Nettoyer la colonne Invoice : mettre en majuscule et supprimer les espaces avant et apres. Sans touchouer aux autres colonnes.

duckdb.sql("""
    CREATE OR REPLACE TABLE retail_clean AS
    SELECT 
        * EXCLUDE (Invoice),
        UPPER(TRIM(Invoice)) AS Invoice
    FROM retail_raw
""")

In [177]:
# Verifier le resultat : afficher une echantillon de 20 factures apres nettoyage.

duckdb.sql("""
    SELECT DISTINCT Invoice
    FROM retail_clean
    LIMIT 20
""").df()

,Invoice
0,C524980
1,C524981
2,C525026
3,C525069
4,525104
5,525103
6,525110
7,525164
8,525223
9,525235


In [178]:
duckdb.sql("""
    CREATE OR REPLACE TABLE d_facture_clean AS
    SELECT Invoice, SalesChannel, PaymentMethod
    FROM retail_clean
""")

In [179]:
duckdb.sql("""
    SELECT *
    FROM d_facture_clean
    LIMIT 10
""").df()

,Invoice,SalesChannel,PaymentMethod
0,489434,Online,Card
1,489434,Online,Card
2,489434,Online,Card
3,489434,Online,Card
4,489434,Online,Card
5,489434,Online,Card
6,489434,Online,Card
7,489434,Online,Card
8,489435,Phone,Invoice
9,489435,Phone,Invoice


Pourquoi une table FACTURE ?
On a vérifié si le canal de vente, le moyen de paiement et le statut sont toujours les mêmes pour toutes les lignes d'une même facture.
Résultat : le canal et le paiement ne changent jamais au sein d'une facture → on les range dans une table FACTURE (une ligne par facture, ça évite de les répéter sur chaque ligne de commande).
Par contre, le statut change dans 713 factures (ex. un produit annulé ou retourné dans une commande) → OrderStatus reste donc dans la table COMMANDES, au niveau de la ligne.

In [180]:
duckdb.sql("""
    SELECT COUNT(*) AS factures_incoherentes
    FROM (
        SELECT Invoice
        FROM retail_raw
        GROUP BY Invoice
        HAVING COUNT(DISTINCT SalesChannel) > 1 
            OR COUNT(DISTINCT PaymentMethod) > 1 
            OR COUNT(DISTINCT OrderStatus) > 1
    )
""").df()

,factures_incoherentes
0,713


In [181]:
duckdb.sql("""
    SELECT
      SUM(CASE WHEN nb_canal    > 1 THEN 1 ELSE 0 END) AS canal_varie,
      SUM(CASE WHEN nb_paiement > 1 THEN 1 ELSE 0 END) AS paiement_varie,
      SUM(CASE WHEN nb_statut   > 1 THEN 1 ELSE 0 END) AS statut_varie
    FROM (
        SELECT Invoice,
               COUNT(DISTINCT SalesChannel)  AS nb_canal,
               COUNT(DISTINCT PaymentMethod) AS nb_paiement,
               COUNT(DISTINCT OrderStatus)   AS nb_statut
        FROM retail_raw
        GROUP BY Invoice
    )
""").df()

,canal_varie,paiement_varie,statut_varie
0,0.0,0.0,713.0


In [182]:
# Vérifier si DiscountRate contient des valeurs vides ou NULL
duckdb.sql("""
    SELECT 
        COUNT(*) AS nb_lignes,
        SUM(CASE WHEN DiscountRate IS NULL THEN 1 ELSE 0 END) AS nb_null,
        SUM(CASE WHEN DiscountRate = '' THEN 1 ELSE 0 END) AS nb_vide
    FROM retail_raw
""").df()

,nb_lignes,nb_null,nb_vide
0,1072707,636.0,0.0


In [183]:
# Modifier definitivement la table d_promotion pour ne garder que les codes promo qui apparaissent plus de 200 000 fois, plus la categoerie VIP.

duckdb.sql("""
    CREATE OR REPLACE TABLE d_promotion AS
    SELECT PromotionCode, DiscountRate
    FROM d_promotion
    WHERE PromotionCode IN (
        SELECT PromotionCode
        FROM d_promotion
        GROUP BY PromotionCode
        HAVING COUNT(*) > 200000
    )
    OR PromotionCode = 'VIP'
""")

In [184]:
duckdb.sql("""
    SELECT PromotionCode, DiscountRate, COUNT(*) AS nb
    FROM d_promotion
    GROUP BY PromotionCode, DiscountRate
    ORDER BY nb DESC
""").df()

,PromotionCode,DiscountRate,nb
0,VIP,0.3,1


In [185]:
duckdb.sql("""
    CREATE OR REPLACE TABLE d_facture_clean AS
    SELECT DISTINCT *
    FROM d_facture_clean
""")

In [186]:
duckdb.sql("""
    CREATE OR REPLACE TABLE d_promotion AS
    SELECT DISTINCT *
    FROM d_promotion
""")

In [187]:
duckdb.sql("SELECT COUNT(*) AS nb_lignes_facture FROM d_facture_clean").df()

,nb_lignes_facture
0,57167


In [188]:
duckdb.sql("SELECT COUNT(*) AS nb_lignes_promotion FROM d_promotion").df()

,nb_lignes_promotion
0,1


In [189]:
# Vérifier les valeurs manquantes (NULL) dans chaque colonne de d_facture_clean
duckdb.sql("""
    SELECT 
        COUNT(*) AS nb_lignes,
        SUM(CASE WHEN Invoice IS NULL THEN 1 ELSE 0 END) AS invoice_null,
        SUM(CASE WHEN SalesChannel IS NULL THEN 1 ELSE 0 END) AS saleschannel_null,
        SUM(CASE WHEN PaymentMethod IS NULL THEN 1 ELSE 0 END) AS paymentmethod_null
    FROM d_facture_clean
""").df()

,nb_lignes,invoice_null,saleschannel_null,paymentmethod_null
0,57167,0.0,3576.0,0.0


In [190]:
# Vérifier s'il y a des doublons sur Invoice dans d_facture_clean
duckdb.sql("""
    SELECT Invoice, COUNT(*) AS nb
    FROM d_facture_clean
    GROUP BY Invoice
    HAVING COUNT(*) > 1
    ORDER BY nb DESC
""").df()

,Invoice,nb
0,544282,2
1,551718,2
2,553487,2
3,554853,2
4,526405,2
...,...,...
3534,528620,2
3535,501889,2
3536,523332,2
3537,560225,2


In [191]:
# Voir le détail d'une facture dupliquée pour comprendre pourquoi
duckdb.sql("""
    SELECT *
    FROM d_facture_clean
    WHERE Invoice = '578935'
""").df()

,Invoice,SalesChannel,PaymentMethod
0,578935,None,Card
1,578935,Online,Card


In [192]:
# Nettoyer d_facture_clean : pour chaque facture, garder la version la plus complète (sans NULL si possible)
duckdb.sql("""
    CREATE OR REPLACE TABLE d_facture_clean AS
    SELECT Invoice, SalesChannel, PaymentMethod
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY Invoice 
                ORDER BY CASE WHEN SalesChannel IS NULL THEN 1 ELSE 0 END
            ) AS rn
        FROM d_facture_clean
    )
    WHERE rn = 1
""")

In [193]:
duckdb.sql("SELECT COUNT(*) AS nb_lignes FROM d_facture_clean").df()

,nb_lignes
0,53628


In [194]:
# Vérifier qu'il n'y a plus de doublins sur INVOICE 

duckdb.sql("""
    SELECT Invoice, COUNT(*) AS nb
    FROM d_facture_clean
    GROUP BY Invoice
    HAVING COUNT(*) > 1
""").df()

,Invoice,nb


In [195]:
duckdb.sql("""
    SELECT COUNT(*) AS nb_null_restant
    FROM d_facture_clean
    WHERE SalesChannel IS NULL
""").df()

,nb_null_restant
0,37


In [196]:
duckdb.sql("""
    SELECT Invoice, COUNT(*) AS nb
    FROM d_facture_clean
    GROUP BY Invoice
    HAVING COUNT(*) > 1
""").df()

,Invoice,nb


In [197]:
duckdb.sql("SELECT COUNT(*) AS nb_lignes FROM d_facture_clean").df()

,nb_lignes
0,53628


In [198]:
# Voir quels PaymentMethod sont associés aux lignes où SalesChannel est vide
duckdb.sql("""
    SELECT PaymentMethod, COUNT(*) AS nb
    FROM d_facture_clean
    WHERE SalesChannel IS NULL
    GROUP BY PaymentMethod
    ORDER BY nb DESC
""").df()

,PaymentMethod,nb
0,PayPal,10
1,Card,10
2,Invoice,9
3,Bank transfer,8


In [199]:
# Vérifier l'association entre PaymentMethod et SalesChannel
duckdb.sql("""
    SELECT PaymentMethod, SalesChannel, COUNT(*) AS nb
    FROM d_facture_clean
    GROUP BY PaymentMethod, SalesChannel
    ORDER BY PaymentMethod, nb DESC
""").df()

,PaymentMethod,SalesChannel,nb
0,Bank transfer,Store,13361
1,Bank transfer,None,8
2,Card,Online,13680
3,Card,None,10
4,Invoice,Phone,13258
5,Invoice,None,9
6,PayPal,Marketplace,13292
7,PayPal,None,10


In [200]:
# Corriger les valeurs manquantes de SalesChannel selon le PaymentMethod correspondant :
# Bank transfer -> Store, Card -> Online, Invoice -> Phone, PayPal -> Marketplace
duckdb.sql("""
    CREATE OR REPLACE TABLE d_facture_clean AS
    SELECT 
        Invoice,
        CASE 
            WHEN SalesChannel IS NULL AND PaymentMethod = 'Bank transfer' THEN 'Store'
            WHEN SalesChannel IS NULL AND PaymentMethod = 'Card' THEN 'Online'
            WHEN SalesChannel IS NULL AND PaymentMethod = 'Invoice' THEN 'Phone'
            WHEN SalesChannel IS NULL AND PaymentMethod = 'PayPal' THEN 'Marketplace'
            ELSE SalesChannel
        END AS SalesChannel,
        PaymentMethod
    FROM d_facture_clean
""")

In [201]:
# Vérifier qu'il n'y a plus aucune valeur NULL dans SalesChannel
duckdb.sql("""
    SELECT COUNT(*) AS nb_null_restant
    FROM d_facture_clean
    WHERE SalesChannel IS NULL
""").df()

,nb_null_restant
0,0


In [202]:
duckdb.sql("""
    SELECT PaymentMethod, SalesChannel, COUNT(*) AS nb
    FROM d_facture_clean
    GROUP BY PaymentMethod, SalesChannel
    ORDER BY PaymentMethod, nb DESC
""").df()

,PaymentMethod,SalesChannel,nb
0,Bank transfer,Store,13369
1,Card,Online,13690
2,Invoice,Phone,13267
3,PayPal,Marketplace,13302


In [203]:
# Réexporter d_facture_clean (sans doublons) en parquet
duckdb.sql("""
    COPY d_facture_clean TO 'd_facture_clean.parquet' (FORMAT PARQUET)
""")

In [204]:
# Réexporter d_promotion (sans doublons) en parquet
duckdb.sql("""
    COPY d_promotion TO 'd_promotion.parquet' (FORMAT PARQUET)
""")